# 剪枝（结构化剪枝）学习笔记

本 notebook 只注重「思路 + 重要代码」，**不带运行输出**。每个流程按 **原理 / 重要 bash 命令 / 重要代码** 三部分整理。

代码取自 `scripts/`；实验 11 多方法对比的脚本在临时目录 `C:/Users/22565/AppData/Local/Temp/yolo_coco128_impr/`（`pruner.py`、`e2_criteria.py`、`e3_sparse.py`、`e4_iterative.py`、`e5_sfp_hrank.py`、`e6_kd.py` 等）。每个代码块标注了来源文件，可回原脚本看完整实现。

## 一、通用准备（所有实验共用，只讲一次）

### 1. 环境安装（ultralytics + torch-pruning）

**原理**

两个核心库：
- `ultralytics`：YOLO11 官方实现，负责模型加载、训练、验证。
- `torch-pruning`：结构化剪枝库，核心是 **DependencyGraph（依赖图）**。

**名词**
- **结构化剪枝 vs 非结构化剪枝**：结构化剪枝删「整条卷积输出通道」，体积/计算量/速度同时受益；非结构化只把单个权重置零，几乎不加速。本项目做的是结构化。
- **依赖图（DependencyGraph）**：记录层与层之间的通道耦合。剪掉某层 conv 的输出通道时，下游 BN 通道数、相连 conv 的输入通道、C2f 的 split/concat 都要跟着一起删，否则形状对不上会报错。`torch-pruning` 的 `build_dependency` + `get_pruning_group` 就是干这个。

本机环境：RTX 5060 Ti、PyTorch 2.11.0+cu128、torch-pruning 1.6.1。

**重要 PowerShell 命令**

```powershell
# 环境依据：README.md 第 62–80 行；当前项目 .venv 中为 Ultralytics 8.4.142、torch-pruning 1.6.1
Set-Location -LiteralPath "C:\Users\22565\OneDrive\Desktop\YOLO剪枝蒸馏初步尝试"

# 仅在需要重建环境时创建虚拟环境；PyTorch 的 CUDA 安装方式应按本机和官方安装源选择
py -3.13 -m venv .venv
.\.venv\Scripts\python.exe -m pip install ultralytics torch torchvision torch-pruning

# 已有环境优先只做检查，避免无意升级依赖
.\.venv\Scripts\python.exe -c "import ultralytics, torch_pruning, torch; print('ultralytics', ultralytics.__version__); print('torch-pruning', torch_pruning.__version__); print('torch', torch.__version__); print('cuda', torch.cuda.is_available())"
```

In [ ]:
# 原项目导入入口摘录
# 出处：scripts/prune_greedy.py 第 25–31 行、scripts/greedy_evaluation.py 第 14–18 行
# 所属阶段：加载张量库、剪枝依赖图、YOLO 模型和自定义训练器

import torch
import torch.nn as nn
import torch_pruning as tp
from ultralytics import YOLO
from ultralytics.models.yolo.detect.train import DetectionTrainer
from ultralytics.nn.modules import C2f, C2PSA

# 阅读项目时先区分两层：
# YOLO(path)       -> Ultralytics 高层封装，提供 train()/val()/export()
# YOLO(path).model -> torch.nn.Module，执行前向、遍历层、构建剪枝依赖图

### 2. 数据下载与划分（coco8 / coco128 / coco2017）

**原理**

数据规模三档：
- `coco8`：8 张，只用于验证流程/环境是否跑通。
- `coco128`：128 张，小规模剪枝实验用（训练快，适合快速试方法）。
- `coco2017`：约 11.8 万训练图 + 5000 验证图，正式实验用。

**为什么 coco128 要自己划分**：coco128 本身没有官方 train/val 划分，脚本按 8:2（102 训练 / 26 验证）固定随机种子 42 切分，保证实验可复现。

**名词**：训练集（更新权重）/ 验证集（只测不练，用来评估精度、选模型）。

注意：coco2017 数据量大，放在仓库外的 `C:/Users/22565/datasets/coco`，不进 OneDrive。

**重要 PowerShell 命令**

```powershell
# 入口出处：scripts/split_coco128.py 第 1–27 行
# 输入：datasets/coco128/images/train2017 与 labels/train2017
# 输出：datasets/coco128_split/images/{train,val} 与 labels/{train,val}
Set-Location -LiteralPath "C:\Users\22565\OneDrive\Desktop\YOLO剪枝蒸馏初步尝试"
.\.venv\Scripts\python.exe scripts\split_coco128.py

# COCO2017 下载定义出处：configs/coco2017.yaml 的 download 段
# 正式数据目录：C:\Users\22565\datasets\coco（不放进 OneDrive 仓库）
```

In [ ]:
# 规范化学习摘录：根据原项目整理，并非逐字复制
# 出处：scripts/split_coco128.py 第 1–27 行
# 所属阶段：固定随机种子切分图片，并同步复制对应标签

from pathlib import Path
import random
import shutil

root = Path("datasets/coco128")
source_images = root / "images/train2017"
source_labels = root / "labels/train2017"
output = root.parent / "coco128_split"

random.seed(42)
images = sorted(source_images.glob("*.jpg"))
random.shuffle(images)
cut = int(len(images) * 0.8)
splits = {"train": images[:cut], "val": images[cut:]}

for split, files in splits.items():
    image_dir = output / "images" / split
    label_dir = output / "labels" / split
    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)

    for image in files:
        shutil.copy2(image, image_dir / image.name)
        label = source_labels / f"{image.stem}.txt"
        if label.exists():
            shutil.copy2(label, label_dir / label.name)

print(f"train={len(splits['train'])}, val={len(splits['val'])}")
print(output)

### 3. 模型下载与加载（yolo11n.pt / yolo11s.pt）

**原理**

- **YOLO11n / YOLO11s**：n = nano（约 2.6M 参数），s = small（约 9.5M 参数）。小模型冗余本来就少，剪枝收益有限——这是本项目「剪得少、掉点多」的根本原因。
- **.pt 里有什么**：ultralytics 的 `.pt` 是一个 dict，含 `model`（网络结构+权重）、`ema`、`optimizer`、`train_args` 等；`YOLO(path)` 负责解析成可用的检测模型。
- **预训练权重**：在 COCO 上已训练好的权重，作为微调/剪枝的起点，比从零训练快很多。

**重要 PowerShell 命令**

```powershell
# API 依据：scripts/run_yolo11s_coco2017_baseline.py 第 108–120 行
# 首次缺少权重时可能访问网络；已有 weights/yolo11s.pt 时直接加载本地文件
Set-Location -LiteralPath "C:\Users\22565\OneDrive\Desktop\YOLO剪枝蒸馏初步尝试"
.\.venv\Scripts\python.exe -c "from ultralytics import YOLO; model = YOLO('models/yolo11n.pt'); print(type(model.model).__name__)"
```

In [ ]:
# 规范化学习摘录：根据原项目整理，并非逐字复制
# 出处：scripts/run_yolo11s_coco2017_baseline.py 第 108–120 行
# 所属阶段：确定配置和权重；缺失时下载；随后构造 YOLO 高层对象

from pathlib import Path

from ultralytics import YOLO
from ultralytics.utils.downloads import attempt_download_asset

ROOT = Path(__file__).resolve().parents[1]
config = ROOT / "configs/coco2017.yaml"
weights = ROOT / "weights/yolo11s.pt"

weights.parent.mkdir(parents=True, exist_ok=True)
downloaded = Path(attempt_download_asset(weights)).resolve()
if not downloaded.is_file():
    raise FileNotFoundError(f"权重下载失败：{downloaded}")

yolo = YOLO(str(downloaded))
model = yolo.model

### 4. 训练与监控进度（train 参数、results.csv / png、best.pt）

**原理**

核心 `train` 参数：
- `epochs`：训练轮数。
- `imgsz`：输入尺寸（常见 640，本项目 coco2017 训练用 512）。
- `batch`：批大小。
- `device`：用哪张卡（0 = 第一张 GPU）。
- `seed`：随机种子，固定后可复现。
- `patience`：早停——连续 N 个 epoch 无提升就提前停。

**best.pt 怎么来**：ultralytics 每个 epoch 在验证集上算 `fitness`（综合 mAP/recall 的分数），最高那轮存为 `best.pt`，最后一轮存 `last.pt`。

**results.csv / results.png**：训练曲线，记录每个 epoch 的 loss 和指标，用来判断有没有过拟合/训练是否正常。

**重要 PowerShell 命令**

```powershell
Set-Location -LiteralPath "C:\Users\22565\OneDrive\Desktop\YOLO剪枝蒸馏初步尝试"

# Ultralytics CLI 示例；入口由当前 .venv\Scripts\yolo.exe 提供
.\.venv\Scripts\yolo.exe detect train data="configs/coco128_split.yaml" model="models/yolo11n.pt" epochs=3 imgsz=640 batch=8 device=0

# 仓库正式训练入口出处：scripts/train_yolo11s_coco2017.py -> main() 第 38–88 行
.\.venv\Scripts\python.exe scripts\train_yolo11s_coco2017.py
```

In [ ]:
# 规范化学习摘录：根据原项目整理，并非逐字复制
# 出处：scripts/train_yolo11s_coco2017.py 第 38–70 行
# 所属阶段：建立带时间戳的运行目录，保存配置，然后调用 YOLO.train()

from datetime import datetime
from pathlib import Path

from ultralytics import YOLO

ROOT = Path(__file__).resolve().parents[1]
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = ROOT / "runs/train/experiment08_yolo11s_coco2017" / stamp
config = ROOT / "configs/coco2017.yaml"
weights = ROOT / "weights/yolo11s.pt"

settings = {
    "model": str(weights),
    "data": str(config),
    "epochs": 30,
    "imgsz": 512,
    "batch": 64,
    "device": 0,
    "workers": 2,
    "seed": 42,
    "deterministic": False,
    "pretrained": True,
    "patience": 10,
}

model = YOLO(str(weights))
model.train(
    data=str(config),
    epochs=settings["epochs"],
    imgsz=settings["imgsz"],
    batch=settings["batch"],
    device=settings["device"],
    workers=settings["workers"],
    seed=settings["seed"],
    deterministic=settings["deterministic"],
    pretrained=settings["pretrained"],
    patience=settings["patience"],
    project=str(run_dir.parent),
    name=stamp,
    exist_ok=True,
    plots=True,
    verbose=True,
)

### 5. 结果指标含义（P / R / mAP50 / mAP50-95 / GMACs / 参数 / FLOPs）

**原理**（老师必问，重点）

- **Precision / Recall**：精确率 = 预测出的框里有多少是真框；召回率 = 真框里有多少被找到。二者是 trade-off。
- **mAP50 / mAP50-95**：在 IoU 阈值 0.5（或 0.5~0.95 取平均）下算的平均精度。mAP50 宽松，**mAP50-95 更严格、是主指标**。
- **参数量 (params)**：所有权重个数，影响体积和显存。
- **GMACs vs FLOPs**：都是计算量单位。**1 GMAC = 2 FLOPs**（一次乘加 MAC 算 2 次浮点运算）。仓库统一用 GMACs。
- **为什么 FLOPs 降了但速度不一定快**：减的量太少（本项目只有 ~1–4%）时，测速波动比收益还大；且真实推理还受内存带宽、算子调度影响，FLOPs 只是近似。

**重要 PowerShell 命令**

```powershell
# Ultralytics CLI 验证；指标出处：scripts/run_yolo11s_coco2017_baseline.py 第 121–174 行
Set-Location -LiteralPath "C:\Users\22565\OneDrive\Desktop\YOLO剪枝蒸馏初步尝试"
.\.venv\Scripts\yolo.exe detect val data="configs/coco128_split.yaml" model="runs/train/experiment02_coco128/weights/best.pt" imgsz=640 batch=8 device=0
```

In [ ]:
# 规范化学习摘录：根据原项目整理，并非逐字复制
# 指标出处：scripts/run_yolo11s_coco2017_baseline.py 第 121–174 行
# 计算量出处：同文件第 149–174 行
# 输入：YOLO 权重和数据配置；输出：检测指标、参数量、GMACs、验证耗时

import copy

yolo = YOLO(str(weights))
metrics = yolo.val(
    data=str(config),
    split="val",
    imgsz=640,
    batch=32,
    device=0,
)

map50 = float(metrics.box.map50)
map50_95 = float(metrics.box.map)
precision = float(metrics.box.mp)
recall = float(metrics.box.mr)
inference_ms = float(metrics.speed["inference"])

raw_model = YOLO(str(weights)).model.float().cuda(0).eval()
example = torch.zeros(1, 3, 640, 640, device="cuda:0")
macs, _ = tp.utils.count_ops_and_params(
    copy.deepcopy(raw_model),
    example_inputs=example,
)
gmacs = float(macs) / 1e9
parameters = sum(parameter.numel() for parameter in raw_model.parameters())

## 二、各实验专属流程

### 敏感度分析（实验 03 coco128 / 实验 09 coco2017）

**原理**

- **目的**：找出哪些层的通道「剪了不疼」，为后续结构化剪枝筛候选层。
- **方法（masking 置零）**：每次从同一个 best.pt 重载，把某一层 L1 重要性最低的约 10% 输出通道权重**置零（不是真删）**，在验证集上测精度下降量 `drop`。
- **L1 重要性**：卷积核权重绝对值越大越重要，用 `weight.abs().mean()` 给每个输出通道打分。
- **关键区别**：masking 只是置零估趋势，**不减少参数量/GMACs**，也不等价于真剪枝（真剪会删通道、改结构）。所以它是「启发式筛选」，不是精确预测。

**名词**：敏感度 = 屏蔽该层后 mAP 下降越多，越敏感、越不能剪。

**重要 PowerShell 命令**

```powershell
# 入口出处：scripts/prune_sensitivity.py -> main() 第 84–150 行
# 输出：reports/experiment03_sensitivity.csv；脚本会覆盖同名旧文件
Set-Location -LiteralPath "C:\Users\22565\OneDrive\Desktop\YOLO剪枝蒸馏初步尝试"
.\.venv\Scripts\python.exe scripts\prune_sensitivity.py `
  --weights "runs\train\experiment02_coco128\weights\best.pt" `
  --data "configs\coco128_split.yaml" `
  --ratio 0.10 `
  --device 0 `
  --output "reports\experiment03_sensitivity.csv"
```

In [ ]:
# 规范化学习摘录：根据原项目整理，并非逐字复制
# 出处：scripts/prune_sensitivity.py
#   find_candidate_layers()：第 36–42 行
#   mask_low_l1_filters()：第 45–55 行
# 所属阶段：找候选卷积层，并屏蔽 L1 最低的输出通道进行敏感度试验

def find_candidate_layers(model: YOLO) -> list[tuple[str, torch.nn.Conv2d]]:
    return [
        (name, module)
        for name, module in model.model.named_modules()
        if isinstance(module, torch.nn.Conv2d) and module.out_channels >= 16
    ]


def mask_low_l1_filters(conv: torch.nn.Conv2d, ratio: float) -> int:
    count = max(1, round(conv.out_channels * ratio))
    count = min(count, conv.out_channels - 1)

    scores = conv.weight.detach().abs().flatten(1).mean(1)
    indices = torch.argsort(scores)[:count]

    with torch.no_grad():
        conv.weight[indices] = 0
        if conv.bias is not None:
            conv.bias[indices] = 0

    return count

### 独立法结构化剪枝（实验 04）

**原理**

- **独立法**：light/balanced/strong 三组分别从**同一个 best.pt 独立**开始，剪不同数量的低敏感层（3/6/9 个），互不影响。
- **真正的结构化剪枝**：这次真的删通道——`DependencyGraph` 自动把下游 BN、相连 conv 输入通道、深度卷积宽度一起删，保证形状对齐。
- **候选层筛选（read_candidates）**：从敏感度 CSV 里挑「低敏感（mAP50-95 drop ≤ 0.005）+ 输出通道 ≥ 64 + 排除首层/注意力/检测头」的层。
- **choose_indices**：按 L1 打分选要删的通道，并把数量**对齐到 8 的倍数**（channel_multiple=8），硬件对 8 的倍数通道更友好。
- **微调**：剪完用 `finetune_pruned_compare.py` 恢复精度（见贪心块里的微调代码）。

**重要 PowerShell 命令**

```powershell
# 入口出处：scripts/prune_independent_compare.py -> main() 第 166–312 行
# 输入：实验02权重、实验03敏感度 CSV；输出：实验04三种剪枝配置及比较表
Set-Location -LiteralPath "C:\Users\22565\OneDrive\Desktop\YOLO剪枝蒸馏初步尝试"
.\.venv\Scripts\python.exe scripts\prune_independent_compare.py `
  --weights "runs\train\experiment02_coco128\weights\best.pt" `
  --data "configs\coco128_split.yaml" `
  --sensitivity "reports\experiment03_sensitivity.csv" `
  --profiles light balanced strong `
  --ratio 0.125 `
  --channel-multiple 8 `
  --device 0
```

In [ ]:
# 规范化学习摘录：根据原项目整理，并非逐字复制
# 出处：scripts/prune_independent_compare.py
#   choose_indices()：第 78–84 行
#   prune_one_layer()：第 87–116 行
# 所属阶段：选择低重要性通道，交给 DependencyGraph 联动删除

def choose_indices(
    conv: torch.nn.Conv2d,
    ratio: float,
    channel_multiple: int,
) -> list[int]:
    desired = max(1, round(conv.out_channels * ratio))
    if conv.out_channels >= channel_multiple * 2:
        desired = max(
            channel_multiple,
            round(desired / channel_multiple) * channel_multiple,
        )
        desired = min(desired, conv.out_channels - channel_multiple)

    scores = conv.weight.detach().abs().flatten(1).mean(1)
    return torch.argsort(scores)[:desired].cpu().tolist()


def prune_one_layer(model, layer_name, ratio, channel_multiple, example):
    for parameter in model.parameters():
        parameter.requires_grad_(True)

    modules = dict(model.named_modules())
    conv = modules.get(layer_name)
    if not isinstance(conv, torch.nn.Conv2d):
        raise RuntimeError(f"找不到可剪卷积层：{layer_name}")

    indices = choose_indices(conv, ratio, channel_multiple)
    graph = tp.DependencyGraph().build_dependency(
        model,
        example_inputs=example,
    )
    group = graph.get_pruning_group(
        conv,
        tp.prune_conv_out_channels,
        idxs=indices,
    )
    if not graph.check_pruning_group(group):
        raise RuntimeError(f"依赖图拒绝剪枝：{layer_name}")

    group.prune()
    with torch.no_grad():
        model(example)  # 剪完立即检查一次前向形状

### 贪心结构化剪枝（实验 05 / 10）

**原理**

- **贪心搜索**：每一步在所有候选层上「试剪」，选**代价最小**的一步接受。代价 = mAP 下降 / GMAC 减少（每省一点计算量掉多少精度）。每步都在当前已剪模型上重新算 L1、重新选，允许同一层重复剪。
- **与独立法的区别**：独立法一次性剪多个层；贪心一步一步来，每步只接受当前最优。
- **保护规则（safe_prune）**：不剪首层(stem)、注意力层、C2f/C2PSA 的 CSP 分块宽度、检测头最终输出（DFL/类别框语义）；每层至少保留初始通道一半。
- **微调恢复（run_finetune）**：剪完精度掉，用少量轮次微调补回来。两个关键点：① 直接 `trainer.model = pruned.model`，绕过 ultralytics 按 YAML 重建（否则剪掉的结构会被重建回去）；② 固定 BN（统计量和仿射参数都不更新），因为小数据会把 BN 统计冲坏。

**名词**：backward_check = 剪完做前向+反向检查，确认梯度有限、结构没坏才接受这一步。

**重要 PowerShell 命令**

```powershell
Set-Location -LiteralPath "C:\Users\22565\OneDrive\Desktop\YOLO剪枝蒸馏初步尝试"

# COCO128 入口出处：scripts/prune_greedy.py -> main() 第 349–381 行
.\.venv\Scripts\python.exe scripts\prune_greedy.py --device 0 --max-steps 20

# COCO2017 入口出处：scripts/prune_greedy_coco2017.py -> main() 第 344–385 行
# 历史实验10所用输入为 weights/baseline/yolo11s_coco2017_best.pt，但该文件当前不在这份项目副本中。
# 以下先检查输入，避免把另一份权重误当成原实验权重；补齐同一权重后才可复现实验10。
$coco2017Weights = "weights\baseline\yolo11s_coco2017_best.pt"
if (-not (Test-Path -LiteralPath $coco2017Weights)) { throw "缺少实验10原始输入权重：$coco2017Weights" }
.\.venv\Scripts\python.exe scripts\prune_greedy_coco2017.py `
  --weights $coco2017Weights `
  --data "configs\coco2017.yaml" `
  --sensitivity "reports\experiment09_yolo11s_coco2017_sensitivity.csv" `
  --device 0 `
  --max-steps 20 `
  --target-reduction 0.20
```

In [ ]:
# 规范化学习摘录：根据原项目整理，并非逐字复制
# 出处：scripts/prune_greedy.py
#   safe_prune()：第 114–161 行
#   贪心代价与选择：第 259–275 行
# 所属阶段：在模型副本上安全试剪；通过保护规则后再比较精度/计算量代价

def safe_prune(model, name, example, initial_channels, ratio=0.125):
    model.eval().float()
    for parameter in model.parameters():
        parameter.requires_grad_(True)

    modules = dict(model.named_modules())
    reverse = {module: module_name for module_name, module in modules.items()}
    root = modules[name]
    indices = choose_indices(root, ratio, 8)
    split_outputs = {
        module.cv1.conv
        for module in model.modules()
        if isinstance(module, (C2f, C2PSA))
    }

    graph = tp.DependencyGraph().build_dependency(model, example_inputs=example)
    group = graph.get_pruning_group(
        root,
        tp.prune_conv_out_channels,
        idxs=indices,
    )
    if not graph.check_pruning_group(group):
        raise ValueError("dependency_group_rejected")

    for dependency, dependent_indices in group:
        module = dependency.target.module
        module_name = reverse.get(module, "")
        prune_output = graph.is_out_channel_pruning_fn(dependency.handler)

        if module in split_outputs and prune_output:
            raise ValueError(f"protect_CSP_chunk_width: {module_name}")
        if module_name == "model.0" or module_name.startswith(("model.0.", "model.10.")):
            raise ValueError(f"protect_stem_or_attention: {module_name}")
        if module_name.startswith("model.23.dfl"):
            raise ValueError(f"protect_DFL: {module_name}")
        if (
            isinstance(module, torch.nn.Conv2d)
            and module_name.startswith("model.23.")
            and module_name.count(".") == 4
            and prune_output
        ):
            raise ValueError(f"protect_detection_output: {module_name}")
        if (
            isinstance(module, torch.nn.Conv2d)
            and prune_output
            and module.out_channels - len(set(dependent_indices))
            < max(8, initial_channels[module_name] // 2)
        ):
            raise ValueError(f"minimum_remaining_channels: {module_name}")

    group.prune()


# trial 是当前模型的副本；measured 是该候选剪完后的验证结果。
gain = (current_gmacs - candidate_gmacs) / baseline_gmacs
drop = current_map - measured["map50_95"]
score = drop / gain

# score 越小，表示每节省一份计算量付出的精度代价越低。

In [ ]:
# 规范化学习摘录：根据原项目整理，并非逐字复制
# 出处：scripts/greedy_evaluation.py
#   _freeze_bn()/FrozenBNTrainer：第 50–64 行
#   run_finetune()：第 66–129 行
# 所属阶段：直接微调已剪结构，并检查 BN、损失和结构没有被意外改变

def _freeze_bn(trainer: DetectionTrainer) -> None:
    for module in trainer.model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()
            for parameter in module.parameters():
                parameter.requires_grad_(False)


class FrozenBNTrainer(DetectionTrainer):
    def _model_train(self):
        super()._model_train()
        _freeze_bn(self)


def _shapes(model: nn.Module) -> dict:
    """记录卷积和 BN 的类型、张量形状及 groups，用于确认剪枝结构未被重建。"""
    return {
        name: (
            type(module).__name__,
            tuple(
                (key, tuple(value.shape))
                for key, value in module.state_dict().items()
            ),
            getattr(module, "groups", None),
        )
        for name, module in model.named_modules()
        if isinstance(module, (nn.Conv2d, nn.BatchNorm2d))
    }


def run_finetune(weights: Path, data: Path, output_dir: Path, device: str, epochs: int = 10) -> Path:
    pruned = YOLO(str(weights))
    expected_shapes = _shapes(pruned.model)

    overrides = {
        "model": str(weights),
        "data": str(data),
        "epochs": epochs,
        "imgsz": 640,
        "batch": 8,
        "device": device,
        "workers": 0,
        "optimizer": "AdamW",
        "lr0": 0.0001,
        "seed": 42,
        "deterministic": True,
        "project": str(output_dir.parent),
        "name": output_dir.name,
        "exist_ok": False,
    }

    trainer = FrozenBNTrainer(overrides=overrides)
    trainer.model = pruned.model  # 保留实际剪枝后的通道结构

    def check_batch(current_trainer):
        if current_trainer.loss is None or not torch.isfinite(current_trainer.loss).all():
            raise RuntimeError("Recovery training produced non-finite loss")

    def check_final(current_trainer):
        if _shapes(current_trainer.model) != expected_shapes:
            raise RuntimeError("Pruned architecture changed during recovery")

    trainer.callbacks["on_pretrain_routine_end"].append(_freeze_bn)
    trainer.callbacks["on_train_batch_end"].append(check_batch)
    trainer.callbacks["on_train_end"].append(check_final)
    trainer.train()

    best = Path(trainer.best)
    if not best.is_file():
        raise RuntimeError("Recovery completed without best.pt")
    return best.resolve()

### 梯度分档剪枝（实验 06）

**原理**

- **梯度重要性（Taylor 类）**：`mean(|W × ∂L/∂W|)`——权重 × 权重梯度 的绝对值。直观理解：这个通道对损失函数的贡献/敏感度，比纯 L1 幅值更能反映「剪掉后损失会怎么变」。
- **分档（tiered）**：复用实验 03 的 L1 masking 敏感度，把层分成 low(≤0.005) / medium(0.005~0.020] / high(>0.020) 三档，A–E 五组按不同比例剪（如 A = 低 12.5%/中 0/高 0，E = 低 25%/中 15%/高 5%）。
- **collect_scores 只统计不改权重**：过一遍全部训练图算梯度打分，但**不更新权重、固定 BN**，并验证模型张量前后一致（fingerprint 相同）。

**重要 PowerShell 命令**

```powershell
# 入口出处：scripts/prune_gradient_tiered.py -> main() 第 347 行至文件结尾
# 输入：实验02基线、实验03敏感度；输出：实验06 A–E 分档结果
Set-Location -LiteralPath "C:\Users\22565\OneDrive\Desktop\YOLO剪枝蒸馏初步尝试"
.\.venv\Scripts\python.exe scripts\prune_gradient_tiered.py --device 0 --epochs 10 --max-map-drop 0.02
```

In [ ]:
# 规范化学习摘录：根据原项目整理，并非逐字复制
# 出处：scripts/prune_gradient_tiered.py
# 对应：collect_scores() 第 59–111 行中的逐批梯度统计核心
# 所属阶段：只收集 Taylor 类通道分数，不调用 optimizer.step()，不修改权重

model.train()
for parameter in model.parameters():
    parameter.requires_grad_(True)
for module in model.modules():
    if isinstance(module, torch.nn.BatchNorm2d):
        module.eval()

scores = {
    row["layer"]: torch.zeros(row["initial_channels"], dtype=torch.float64)
    for row in tiers
}
images = 0

for batch in loader:
    batch = {
        key: value.to(device) if isinstance(value, torch.Tensor) else value
        for key, value in batch.items()
    }
    batch["img"] = batch["img"].float() / 255.0
    batch_size = batch["img"].shape[0]

    model.zero_grad(set_to_none=True)
    loss, _ = model(batch)
    loss = loss.sum() / batch_size
    loss.backward()

    for name, total in scores.items():
        weight = modules[name].weight
        if weight.grad is None:
            raise RuntimeError(f"missing gradient: {name}")
        value = (
            (weight.detach() * weight.grad.detach())
            .abs()
            .flatten(1)
            .mean(1)
        )
        total.add_(value.double().cpu(), alpha=batch_size)

    images += batch_size

scores = {name: (total / images).tolist() for name, total in scores.items()}

### 多方法对比：重要性准则（实验 11 coco128）

**原理**

- **目的**：同一基线（exp02 best.pt）、同样 9 个安全层、同样约 10%/层 的剪幅下，横向对比 6 种重要性准则。**控制变量后，差异才纯来自打分函数本身**。
- **安全层（SAFE_LAYERS）**：实验 03 敏感度验证过的 9 个「剪了不疼」的层（低敏感 + 输出通道 ≥ 64 + 排除首层/注意力/检测头 + 排除 TP-1.6.1 残差输出 bug 层 model.13.m.0.cv2.conv）。YOLO11n 冗余小，总剪幅被锁在 ~3.6% 参数附近——这是所有方法「提升不明显」的根本原因，不是方法不行。
- **统一框架**：所有准则都在同一批候选层上给每个**输出通道**打重要性分，剪掉分数最低的一批；换打分函数 = 换方法。

| 准则 | 原理 |
|---|---|
| L1 | 卷积核绝对值均值小 → 不重要 |
| L2 | 卷积核 L2 范数小 → 不重要 |
| BN-γ | 该 conv 跟随的 BN 缩放因子 \|γ\| 小 → 不重要 |
| FPGM | 离同类通道的几何中位数近 → 冗余（可被替代） |
| Taylor | mean\|W × dL/dW\| 小 → 该通道对损失不敏感 |
| HRank | 输出特征图矩阵秩低 → 信息少 |

**结论：Taylor > FPGM > L1 ≈ L2 ≫ BN-γ**（raw：0.336 / 0.326 / 0.314 / 0.315 / 0.012）。「看梯度敏感度」比「看幅值」更准。BN-γ **不先做稀疏训练直接剪是灾难**——BN 的 γ 本身不稀疏，按它排序 ≈ 乱剪。

**名词**：几何中位数 = 与所有同类通道距离和最小的点；HRank 的秩 = 特征矩阵 SVD 后有效奇异值个数。

**重要 PowerShell 命令**

```powershell
# 命令出处：C:\Users\22565\AppData\Local\Temp\yolo_coco128_impr 中的历史实验11脚本
# 性质：这些脚本不属于仓库正式 scripts/，当前仍存在，但临时目录以后可能被清理。
$experiment11 = "C:\Users\22565\AppData\Local\Temp\yolo_coco128_impr"
$projectPython = "C:\Users\22565\OneDrive\Desktop\YOLO剪枝蒸馏初步尝试\.venv\Scripts\python.exe"
Set-Location -LiteralPath $experiment11

& $projectPython .\e0_heldout.py
& $projectPython .\e2_criteria.py
& $projectPython .\e5_sfp_hrank.py hrank
```

In [ ]:
# 规范化学习摘录：根据历史实验代码整理，并非逐字复制
# 出处：C:\Users\22565\AppData\Local\Temp\yolo_coco128_impr\pruner.py
#   L1/L2/BN/FPGM：第 208–256 行
#   Taylor：第 293–330 行
# 性质：实验11临时实现，不属于仓库正式 scripts/

def score_l1(model):
    return {
        name: module.weight.detach().abs().mean(dim=(1, 2, 3)).cpu().numpy()
        for name, module in candidate_layers(model)
    }


def score_l2(model):
    return {
        name: module.weight.detach().pow(2).sum(dim=(1, 2, 3)).sqrt().cpu().numpy()
        for name, module in candidate_layers(model)
    }


def score_bn(model):
    scores = {}
    modules = dict(model.named_modules())
    for name, conv in candidate_layers(model):
        parent = modules.get(name.rsplit(".", 1)[0])
        bn = getattr(parent, "bn", None)
        if isinstance(bn, nn.BatchNorm2d) and bn.weight.numel() == conv.out_channels:
            scores[name] = bn.weight.detach().abs().cpu().numpy()
        else:
            scores[name] = conv.weight.detach().abs().mean(dim=(1, 2, 3)).cpu().numpy()
    return scores


def score_fpgm(model):
    scores = {}
    for name, conv in candidate_layers(model):
        weight = conv.weight.detach().float().reshape(conv.out_channels, -1)
        squared_norm = weight.pow(2).sum(1, keepdim=True)
        squared_distance = (
            squared_norm + squared_norm.T - 2 * weight @ weight.T
        ).clamp_min(0)
        scores[name] = squared_distance.sqrt().sum(dim=1).cpu().numpy()
    return scores


def score_taylor(model, split_dir, n_batches=13):
    """历史实验11使用预测张量平方和作为代理损失。"""
    model.train()
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()
    for parameter in model.parameters():
        parameter.requires_grad_(True)

    accumulated = {
        name: torch.zeros(module.out_channels, device="cpu")
        for name, module in candidate_layers(model)
    }
    seen = 0

    def tensors(value):
        if isinstance(value, torch.Tensor):
            yield value
        elif isinstance(value, dict):
            for item in value.values():
                yield from tensors(item)
        elif isinstance(value, (tuple, list)):
            for item in value:
                yield from tensors(item)

    for images in TrainImageBatcher(split_dir):
        if seen >= n_batches:
            break
        model.zero_grad(set_to_none=True)
        predictions = model(images.to(DEVICE))
        proxy_loss = sum(
            tensor.float().square().mean()
            for tensor in tensors(predictions)
        )
        proxy_loss.backward()

        with torch.no_grad():
            for name, module in candidate_layers(model):
                if module.weight.grad is not None:
                    accumulated[name] += (
                        module.weight.detach() * module.weight.grad.detach()
                    ).abs().mean(dim=(1, 2, 3)).cpu()
        seen += 1

    return {name: (value / max(seen, 1)).numpy() for name, value in accumulated.items()}

In [ ]:
# 规范化学习摘录：根据历史实验代码整理，并非逐字复制
# 出处：C:\Users\22565\AppData\Local\Temp\yolo_coco128_impr\e2_criteria.py 第 25–63 行
# 所属阶段：固定基线、候选层和剪幅，只替换通道重要性准则
# 性质：实验11临时实现，不属于仓库正式 scripts/

model = fresh_base(base)
scores = {
    "l1": score_l1(model),
    "l2": score_l2(model),
    "bn": score_bn(model),
    "fpgm": score_fpgm(model),
}
scores["taylor"] = score_taylor(model, split_dir, n_batches=13)
baseline_parameters, baseline_gmacs = stats_of(model)

control_best = finetune(base, "e2_ft_control", epochs=50, lr0=0.001)
val_map(control_best, "e2_control_val2017", data=COCO2017_VAL_YAML)

for criterion_name in ("l1", "l2", "bn", "fpgm", "taylor"):
    candidate_model = fresh_base(base)
    plan = uniform_plan(
        candidate_model,
        scores[criterion_name],
        ratio=0.10,
    )
    candidate_model, completed, skipped = apply_plan(candidate_model, plan)

    checkpoint = SCRATCH / f"e2_{criterion_name}_pruned.pt"
    save_pruned(candidate_model, checkpoint, base)

    val_map(
        checkpoint,
        f"e2_{criterion_name}_raw2017",
        data=COCO2017_VAL_YAML,
        batch=8,
    )
    best = finetune(
        checkpoint,
        f"e2_ft_{criterion_name}",
        epochs=50,
        lr0=0.001,
    )
    val_map(best, f"e2_{criterion_name}_val2017", data=COCO2017_VAL_YAML)

In [ ]:
# 规范化学习摘录：根据历史实验代码整理，并非逐字复制
# 出处：C:\Users\22565\AppData\Local\Temp\yolo_coco128_impr\e5_sfp_hrank.py 第 83–149 行
# 所属阶段：用 hook 收集特征图，再按每个输出通道的特征矩阵秩排序
# 性质：实验11临时实现，不属于仓库正式 scripts/

model = fresh_base(base).eval()
candidates = candidate_layers(model)
sample_count = 512
features = {name: [] for name, _ in candidates}
handles = []


def make_hook(name):
    def hook(_module, _inputs, output):
        feature = output.detach().float()
        if feature.dim() != 4:
            return

        batch, channels, height, width = feature.shape
        feature = feature.permute(1, 0, 2, 3).reshape(
            channels,
            batch,
            height * width,
        )
        if height * width > sample_count:
            indices = torch.linspace(
                0,
                height * width - 1,
                sample_count,
                dtype=torch.long,
            )
            feature = feature[:, :, indices]
        features[name].append(feature)

    return hook


modules = dict(model.named_modules())
for name, _ in candidates:
    handles.append(modules[name].register_forward_hook(make_hook(name)))

with torch.no_grad():
    for images in TrainImageBatcher(SCRATCH / "coco128_split"):
        model(images.to(DEVICE))

for handle in handles:
    handle.remove()

scores = {}
for name, _ in candidates:
    matrix = torch.cat(features[name], dim=1)
    ranks = torch.zeros(matrix.shape[0])
    for channel in range(matrix.shape[0]):
        singular_values = torch.linalg.svdvals(matrix[channel])
        if singular_values.numel() and singular_values[0] > 0:
            ranks[channel] = (singular_values > 0.9 * singular_values[0]).sum()
    scores[name] = ranks.numpy()

### 恢复手段对比（实验 11 续 coco128）

**原理**

剪完精度必然掉，各手段的思路是「怎么把精度补回来」：

| 手段 | 原理 | 结果（held-out val2017） |
|---|---|---|
| 微调 | 剪完继续训 50ep，让模型适应新结构 | 标准做法，L1 raw 0.314 → 0.340 |
| 软剪枝 SFP | 先**归零**通道（结构还在）训练几轮让模型适应，再一次性硬剪 | **本轮最佳 0.357**（剪幅最小 2.6%） |
| Network Slimming | 先给 BN-γ 加 L1 惩罚稀疏训练 150ep，再按 \|γ\| 剪 | 对 YOLO11n **失败**（γ 压不稀疏，raw 0.004） |
| KD 蒸馏 | 剪后学生向未剪教师的输出 logits 对齐 | 无增益（0.333 vs 普通微调 0.340） |
| 迭代剪枝 | 3 轮 {剪 ~4%/层 → 微调 30ep} | 0.329，不比一次性剪好 |
| 量化 INT8 | 权重/激活转低精度 | 未跑（缺 onnx/tensorrt） |

**结果速查**（YOLO11n，held-out val2017，基准 0.386）：

| 方法 | raw | 微调后 | 参数降幅 |
|---|---:|---:|---:|
| L1 | 0.314 | 0.340 | 3.7% |
| L2 | 0.315 | 0.340 | 3.7% |
| FPGM | 0.326 | 0.337 | 3.7% |
| **Taylor** | **0.336** | **0.344** | 3.7% |
| BN-γ（无稀疏） | 0.012 | 0.195 | 3.7% |
| Network Slimming | 0.004 | 0.096 | 3.7% |
| 迭代剪枝 | — | 0.329 | 4.0% |
| **SFP** | — | **0.357** | 2.6% |
| HRank | — | 0.319 | 3.7% |
| KD 蒸馏 | — | 0.333 | 3.7% |

**评测口径（重要教训）**：必须用 **held-out val2017**（coco2017 的 5000 张验证集）。coco128 的 26-val 会系统性高估 ~0.24 且**低估剪枝损失**——仓库早先「independent > greedy」的结论就是被这个口径污染（E0 重评发现）。

**后续方向（按杠杆排序）**：① 量化 INT8（体积 −75%、速度 2~3×、几乎不掉点，最大免费杠杆）；② 对 C2f split/concat 做成对结构化剪枝（剪幅可从 3.6% 提到 30%+）；③ 高剪枝率 + COCO2017 长微调；④ 换更大的 YOLO11s/m（冗余多、剪枝收益更明显）。

**重要 PowerShell 命令**

```powershell
# 命令出处：C:\Users\22565\AppData\Local\Temp\yolo_coco128_impr 中的历史实验11脚本
# 性质：以下命令依赖临时目录中的 lab.py、pruner.py、数据软副本及历史中间权重。
$experiment11 = "C:\Users\22565\AppData\Local\Temp\yolo_coco128_impr"
$projectPython = "C:\Users\22565\OneDrive\Desktop\YOLO剪枝蒸馏初步尝试\.venv\Scripts\python.exe"
Set-Location -LiteralPath $experiment11

& $projectPython .\e5_sfp_hrank.py sfp
& $projectPython .\e3_sparse.py
& $projectPython .\e4_iterative.py
& $projectPython .\e6_kd.py
& $projectPython .\e7_quant.py
```

In [ ]:
# 规范化学习摘录：根据历史实验代码整理，并非逐字复制
# 出处：C:\Users\22565\AppData\Local\Temp\yolo_coco128_impr\e5_sfp_hrank.py 第 14–78 行
# 所属阶段：逐轮归零低 L2 通道并短暂微调，最后执行一次真实结构化剪枝
# 性质：实验11临时实现，不属于仓库正式 scripts/

model = YOLO(str(base)).model.float().cuda()
masks: dict[str, set[int]] = {}

overrides = {
    "model": str(base),
    "data": str(COCO128_YAML),
    "epochs": 4,
    "imgsz": 640,
    "batch": 8,
    "device": 0,
    "workers": 0,
    "seed": 42,
    "plots": False,
    "lr0": 0.001,
}

for round_index in range(5):
    scores = score_l2(model)

    for name, _ in candidate_layers(model):
        alive = [
            index
            for index in range(len(scores[name]))
            if index not in masks.get(name, set())
        ]
        remove_count = max(1, int(len(alive) * 0.02))
        order = torch.argsort(torch.as_tensor(scores[name], dtype=torch.float32))
        selected = [index for index in order.tolist() if index in alive][:remove_count]
        masks.setdefault(name, set()).update(selected)

        with torch.no_grad():
            dict(model.named_modules())[name].weight[selected] = 0.0

    def rezero(trainer):
        """每个训练 batch 后再次清零，避免优化器把被屏蔽通道更新回来。"""
        with torch.no_grad():
            modules = dict(trainer.model.named_modules())
            for layer_name, indices in masks.items():
                if layer_name in modules:
                    modules[layer_name].weight[list(indices)] = 0.0

    trainer = DetectionTrainer(overrides=overrides)
    trainer.model = model
    trainer.callbacks["on_train_batch_end"].append(rezero)
    trainer.train()
    model = YOLO(str(trainer.best)).model.float().cuda()

hard_pruning_plan = {
    name: sorted(indices)
    for name, indices in masks.items()
    if indices
}
hard_pruned_model = fresh_base(base)
hard_pruned_model, completed, skipped = apply_plan(
    hard_pruned_model,
    hard_pruning_plan,
)
checkpoint = SCRATCH / "e5_sfp_pruned.pt"
save_pruned(hard_pruned_model, checkpoint, base)
best = finetune(checkpoint, "e5_ft_sfp", epochs=50, lr0=0.001)

In [ ]:
# 规范化学习摘录：根据历史实验代码整理，并非逐字复制
# 出处：C:\Users\22565\AppData\Local\Temp\yolo_coco128_impr\e3_sparse.py 第 12–77 行
# 所属阶段：先把 BN gamma 推向稀疏，再按 |gamma| 排序剪枝
# 性质：实验11临时实现，不属于仓库正式 scripts/

class SparseCriterion:
    def __init__(self, base_criterion, model, sparsity_strength=0.001):
        self.base = base_criterion
        self.sparsity_strength = sparsity_strength
        self.batch_norms = [
            module
            for name, module in model.named_modules()
            if isinstance(module, nn.BatchNorm2d)
            and not name.startswith("model.23")
        ]

    def __call__(self, predictions, batch):
        detection_loss, loss_items = self.base(predictions, batch)
        gamma_penalty = torch.zeros(
            (),
            device=detection_loss.device,
            dtype=detection_loss.dtype,
        )
        for batch_norm in self.batch_norms:
            gamma_penalty = gamma_penalty + batch_norm.weight.abs().sum()

        total_loss = detection_loss + self.sparsity_strength * gamma_penalty
        return total_loss, loss_items


model = YOLO(str(base))
model.model = model.model.float().cuda()
overrides = {
    "model": str(base),
    "data": str(COCO128_YAML),
    "epochs": 150,
    "imgsz": 640,
    "batch": 8,
    "device": 0,
    "workers": 0,
    "seed": 42,
    "lr0": 0.001,
    "patience": 150,
}
trainer = DetectionTrainer(overrides=overrides)
trainer.model = model.model
model.model.args = trainer.args
if model.model.criterion is None:
    model.model.criterion = model.model.init_criterion()

model.model.criterion = SparseCriterion(
    model.model.criterion,
    model.model,
    sparsity_strength=0.1,
)
trainer.train()

sparse_last = Path(trainer.last).resolve()
sparse_model = fresh_base(sparse_last)
plan = uniform_plan(
    sparse_model,
    score_bn(sparse_model),
    ratio=0.10,
)
sparse_model, completed, skipped = apply_plan(sparse_model, plan)

In [ ]:
# 规范化学习摘录：根据历史实验代码整理，并非逐字复制
# 出处：C:\Users\22565\AppData\Local\Temp\yolo_coco128_impr\e6_kd.py 第 17–99 行
# 所属阶段：在原检测损失上增加未剪教师与剪后学生的检测头响应 MSE
# 性质：实验11临时实现；它不同于 distillation.ipynb 的 DINOv2 特征蒸馏

class KDCriterion:
    def __init__(self, base_criterion, teacher, weight=1.0):
        self.base = base_criterion
        self.teacher = teacher
        self.weight = weight

        self.teacher.eval()
        for parameter in self.teacher.parameters():
            parameter.requires_grad_(False)
        self.teacher_head = self.teacher.model[-1]

    @staticmethod
    def flatten_tensors(value):
        if isinstance(value, torch.Tensor):
            return [value]
        if isinstance(value, dict):
            return [item for item in value.values() if isinstance(item, torch.Tensor)]
        if isinstance(value, (tuple, list)):
            flattened = []
            for item in value:
                flattened.extend(KDCriterion.flatten_tensors(item))
            return flattened
        return []

    def __call__(self, student_predictions, batch):
        detection_loss, loss_items = self.base(student_predictions, batch)

        with torch.no_grad():
            self.teacher_head.train()
            teacher_predictions = self.teacher.predict(batch["img"])
            self.teacher_head.eval()

        student_outputs = self.flatten_tensors(student_predictions)
        teacher_outputs = self.flatten_tensors(teacher_predictions)

        distillation_loss = torch.zeros((), device=batch["img"].device)
        matched_outputs = 0
        for student_output, teacher_output in zip(student_outputs, teacher_outputs):
            if student_output.shape == teacher_output.shape:
                distillation_loss += F.mse_loss(
                    student_output.float(),
                    teacher_output.float(),
                )
                matched_outputs += 1

        if matched_outputs:
            distillation_loss /= matched_outputs

        return detection_loss + self.weight * distillation_loss, loss_items


# 将自定义损失接入 Ultralytics 训练器
teacher = YOLO(str(base)).model.float().cuda().eval()
student_checkpoint = SCRATCH / "e2_l1_pruned.pt"
student = YOLO(str(student_checkpoint))
student.model = student.model.float().cuda()

overrides = {
    "model": str(student_checkpoint),
    "data": str(COCO128_YAML),
    "epochs": 50,
    "imgsz": 640,
    "batch": 8,
    "device": 0,
    "workers": 0,
    "seed": 42,
    "lr0": 0.001,
    "patience": 30,
}
trainer = DetectionTrainer(overrides=overrides)
trainer.model = student.model
student.model.args = trainer.args
if student.model.criterion is None:
    student.model.criterion = student.model.init_criterion()

student.model.criterion = KDCriterion(
    student.model.criterion,
    teacher,
    weight=1.0,
)
trainer.train()

### 测速与计算量统计（GMACs / 参数 / 前向耗时）

**原理**

- **GMACs / 参数**：用 torch-pruning 的 `count_ops_and_params` 在**未融合(unfused)**结构上统计，保证剪枝前后口径一致。
- **测速**：`fuse()` 融合 Conv+BN（推理常用、更快），batch=1、640×640，预热 30 次后测量 250 次取**中位数**（抗偶发波动）。用 CUDA Event 计时 + 同步。
- **为什么「FLOPs 降了但没加速」**：减的量太少（~1–4%）时，轮间波动比收益还大；真实推理还受内存带宽/算子调度影响。

**名词**：fuse（融合）= 把 Conv 后的 BN 合并进 Conv 权重，推理时少一层计算。

**重要 PowerShell 命令**

```powershell
# 无独立测速入口。
# 正式实现出处：scripts/greedy_evaluation.py -> benchmark() 第 152–223 行
# prune_greedy.py 会在搜索和微调完成后自动调用 benchmark()（第 336 行）。
```

In [ ]:
# 规范化学习摘录：根据原项目整理，并非逐字复制
# 出处：scripts/greedy_evaluation.py -> benchmark() 第 152–223 行
# 所属阶段：统一统计参数/GMACs，融合 Conv+BN，预热并测量 250 次网络前向

import copy
import statistics
from pathlib import Path


def benchmark_one(weights_path: Path, device: str = "0") -> dict[str, float]:
    target = torch.device(f"cuda:{device}")
    example = torch.zeros(
        1,
        3,
        640,
        640,
        device=target,
        dtype=torch.float32,
    )

    raw_model = YOLO(str(weights_path)).model.to(target).float().eval()
    parameters = sum(parameter.numel() for parameter in raw_model.parameters())
    macs, _ = tp.utils.count_ops_and_params(
        copy.deepcopy(raw_model),
        example_inputs=example,
    )
    timed_model = raw_model.fuse(verbose=False).eval()

    samples_ms = []
    with torch.inference_mode():
        for _ in range(30):
            timed_model(example)
        torch.cuda.synchronize(target)

        for _round in range(5):
            for _ in range(50):
                start = torch.cuda.Event(enable_timing=True)
                end = torch.cuda.Event(enable_timing=True)
                start.record()
                timed_model(example)
                end.record()
                torch.cuda.synchronize(target)
                samples_ms.append(start.elapsed_time(end))

    return {
        "parameters": float(parameters),
        "gmacs": float(macs) / 1e9,
        "latency_median_ms": statistics.median(samples_ms),
        "latency_p90_ms": float(torch.quantile(torch.tensor(samples_ms), 0.9)),
    }

# 这里统计的是 batch=1、FP32、融合后纯网络前向，不含读图、预处理和 NMS。